In [1]:
# 数据读取与初检
from pathlib import Path
import pandas as pd

# 推断数据路径（在scripts目录下相对../raw）
data_path_candidates = [
    Path("../raw/gkx_20201231.csv"),
    Path("../../raw/gkx_20201231.csv"),
    Path("Assignment1/raw/gkx_20201231.csv"),
]

for candidate in data_path_candidates:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError("未找到 gkx_20201231.csv，请检查路径")

# 读取数据
raw_df = pd.read_csv(data_path, low_memory=False)

# 日期转换（按YYYYMMDD解析字符串，避免被当作秒）
if "DATE" in raw_df.columns:
    raw_df["DATE"] = pd.to_datetime(raw_df["DATE"].astype(str), format="%Y%m%d", errors="coerce")
    raw_df = raw_df.sort_values("DATE")

# 基础信息
n_rows, n_cols = raw_df.shape
print(f"数据维度: {n_rows} 行, {n_cols} 列")
if "DATE" in raw_df.columns:
    print(
        "日期范围:",
        raw_df["DATE"].min(),
        "→",
        raw_df["DATE"].max(),
    )

print("\n列名预览:", list(raw_df.columns[:50]), "...")

# 缺失率Top 10
missing_rate = raw_df.isna().mean().sort_values(ascending=False)
print("\n缺失率Top10:\n", missing_rate.head(10))

# 数值列基本统计（前5列示例）
numeric_cols = raw_df.select_dtypes(include="number").columns
if len(numeric_cols) > 0:
    display(raw_df[numeric_cols[:5]].describe(percentiles=[0.01, 0.5, 0.99]))

# 前几行数据预览
display(raw_df.head())


数据维度: 4345508 行, 101 列
日期范围: 1926-01-30 00:00:00 → 2020-12-31 00:00:00

列名预览: ['permno', 'DATE', 'mvel1', 'RET', 'prc', 'SHROUT', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'mve0', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof'] ...

缺失率Top10:
 realestate    0.756666
rd_sale       0.673821
rd_mve        0.668240
secured       0.637079
stdcf         0.636829
stdacc        0.636829
roavol        0.531064
orgcap        0.521404
grltnoa       0.520301
pchsaleinv    0.509615
dtype: float64


,permno,mvel1,RET,prc,SHROUT
count,4.345508e+06,4.341591e+06,4.345508e+06,4.324990e+06,4.345508e+06
mean,5.325557e+04,1.247499e+06,1.083914e-02,3.114100e+01,4.502779e+04
std,2.874066e+04,6.074550e+06,1.740540e-01,1.347706e+03,2.316850e+05
min,1.000000e+04,0.000000e+00,-1.988095e+00,7.800000e-03,0.000000e+00
1%,1.025300e+04,9.494938e+02,-3.863640e-01,2.656250e-01,2.000000e+02
50%,5.669600e+04,8.348208e+04,0.000000e+00,1.370000e+01,8.240000e+03
99%,9.268100e+04,2.472807e+07,5.521721e-01,1.220000e+02,6.121904e+05
max,9.343600e+04,1.735365e+08,2.400000e+01,3.478150e+05,2.920640e+07


,permno,DATE,mvel1,RET,prc,SHROUT,beta,betasq,chmom,dolvol,...,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,sic2,bm,bm_ia
0,10006,1926-01-30,65400.000,0.032732,110.25,600,NaN,NaN,NaN,NaN,...,0.006857,NaN,NaN,NaN,NaN,NaN,0.000066,NaN,NaN,NaN
338,13960,1926-01-30,1629.375,0.272727,1.75,1185,NaN,NaN,NaN,NaN,...,0.181818,NaN,NaN,NaN,NaN,NaN,0.000058,NaN,NaN,NaN
337,13952,1926-01-30,11638.375,0.081272,38.25,329,NaN,NaN,NaN,NaN,...,0.031634,NaN,NaN,NaN,NaN,NaN,0.000006,NaN,NaN,NaN
336,13944,1926-01-30,6256.250,0.202797,43.00,175,NaN,NaN,NaN,NaN,...,0.041958,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
335,13936,1926-01-30,5015.000,-0.059322,55.50,85,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000019,NaN,NaN,NaN


## 基于初检结果的下一步计划
上面的输出显示：
- 数据量大（约434万行、101列），按`DATE`排序后日期字段格式化为时间戳。
- 缺失较严重的特征（如realestate、rd_sale等）缺失率>50%，需要缺失值策略（删除列或填补）。
- 价格/市值等列存在长尾与极端值，后续正则化模型需标准化。

因此，下一步先做“初步清洗与特征准备”：删除指定无效列、明确目标`RET`、抽取特征列、查看特征缺失率和行级缺失分布，为后续缺失处理与标准化做决策。

In [2]:
# 初步清洗与特征准备
import numpy as np

cols_del = ['SHROUT', 'mve0', 'prc', 'permno', 'DATE', 'sic2']
target_col = 'RET'

if target_col not in raw_df.columns:
    raise KeyError('RET 不在数据列中，请检查数据字典')

# 复制并按日期排序，避免未来信息泄漏
work_df = raw_df.copy()
if 'DATE' in work_df.columns:
    work_df = work_df.sort_values('DATE')

# 目标与特征列划分
drop_cols = [c for c in cols_del if c in work_df.columns]
feature_cols = [c for c in work_df.columns if c not in drop_cols + [target_col]]
features = work_df[feature_cols]
y = work_df[target_col]

print(f"保留特征列数: {len(feature_cols)}（删除列: {drop_cols}；目标列: {target_col}）")

# 特征缺失率（Top10）
feature_missing = features.isna().mean().sort_values(ascending=False)
print("\n特征缺失率Top10:")
print(feature_missing.head(10))

# 行级缺失占比分布（抽样以节省计算）
sample_n = min(len(features), 50000)
row_missing_frac = features.sample(sample_n, random_state=42).isna().mean(axis=1)
print("\n行级缺失占比分布（抽样" + str(sample_n) + "行）：")
print(row_missing_frac.describe(percentiles=[0.5, 0.9, 0.99])) 

# 示例行预览
display(features.head())
display(y.head())


保留特征列数: 94（删除列: ['SHROUT', 'mve0', 'prc', 'permno', 'DATE', 'sic2']；目标列: RET）

特征缺失率Top10:
realestate    0.756666
rd_sale       0.673821
rd_mve        0.668240
secured       0.637079
stdcf         0.636829
stdacc        0.636829
roavol        0.531064
orgcap        0.521404
grltnoa       0.520301
pchsaleinv    0.509615
dtype: float64

行级缺失占比分布（抽样50000行）：
count    50000.000000
mean         0.333408
std          0.335975
min          0.000000
50%          0.170213
90%          0.797872
99%          0.925532
max          0.978723
dtype: float64


,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,mom6m,mom12m,...,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,bm,bm_ia
0,65400.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.006857,NaN,NaN,NaN,NaN,NaN,0.000066,NaN,NaN
27,21621.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.040000,NaN,NaN,NaN,NaN,NaN,0.000001,NaN,NaN
247,132358.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.012978,NaN,NaN,NaN,NaN,NaN,0.000028,NaN,NaN
1,11200.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.018018,NaN,NaN,NaN,NaN,NaN,0.000003,NaN,NaN
2,23400.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.004158,NaN,NaN,NaN,NaN,NaN,0.000023,NaN,NaN


0      0.032732
27     0.000000
247    0.065903
1      0.017857
2      0.161667
Name: RET, dtype: float64

## 缺失值处理策略与下一步
上面结果显示：
- 个别特征缺失率 >60%（realestate、rd_sale、rd_mve 等），直接保留会降低有效样本维度。
- 行级缺失分布中位数约17%，90分位约80%，存在高缺失行；直接全量行删除会损失太多，但应剔除缺失极端行。

因此，先进行：
1) 列删：剔除缺失率 >0.6 的特征。
2) 行筛：剔除行缺失占比 >0.9 的极端行。
3) 数值填补：对剩余特征用中位数填补（后续可改为按期截面中位数以降低偏移）。
4) 保留列名并输出新形状，为后续标准化和时间序列CV做准备。


In [3]:
# 缺失值初步处理（列删+行筛，不在全量上填补，避免泄露）

# 1) 列删：缺失率>0.6 的特征
high_missing_cols = feature_missing[feature_missing > 0.6].index.tolist()
features_step1 = features.drop(columns=high_missing_cols)
print(f"删除高缺失列 {len(high_missing_cols)} 个，剩余特征列 {features_step1.shape[1]} 个")

# 2) 行筛：缺失占比>0.9 的行剔除
row_missing_frac_step1 = features_step1.isna().mean(axis=1)
row_keep_mask = row_missing_frac_step1 <= 0.9
features_step2 = features_step1.loc[row_keep_mask]
y_step2 = y.loc[row_keep_mask]
print(f"剔除缺失>90%行 {(~row_keep_mask).sum()} 条，剩余 {features_step2.shape[0]} 行")

# 保留基础特征与目标（未填补，折内再fit imputer）
X_base = features_step2
print("基础特征形状:", X_base.shape)
print("目标形状:", y_step2.shape)

# 预览
print("\n缺失后特征示例:")
display(X_base.head())
print("\n对应目标示例:")
display(y_step2.head())


删除高缺失列 6 个，剩余特征列 88 个
剔除缺失>90%行 54195 条，剩余 4291313 行
基础特征形状: (4291313, 88)
目标形状: (4291313,)

缺失后特征示例:


,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,mom6m,mom12m,...,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,bm,bm_ia
504,5312.50,NaN,NaN,NaN,NaN,NaN,NaN,0.011905,NaN,NaN,...,NaN,0.025226,2.906579e-06,0.073171,0.031768,1.058389,4.779582,1.957494e-08,NaN,NaN
503,22110.00,NaN,NaN,NaN,NaN,NaN,NaN,0.002278,NaN,NaN,...,NaN,0.009646,1.011980e-07,0.031142,0.009619,1.133733,6.069652,1.347701e-08,NaN,NaN
508,27487.50,NaN,NaN,NaN,NaN,NaN,NaN,0.001344,NaN,NaN,...,NaN,0.012694,1.442342e-07,0.019074,0.010184,1.223397,5.798691,8.400000e-01,NaN,NaN
502,9834.00,NaN,NaN,NaN,7.280077,NaN,NaN,-0.083333,NaN,NaN,...,NaN,0.015453,3.919852e-06,0.062500,0.023225,0.770209,0.453796,8.801688e-08,NaN,NaN
501,2535.75,NaN,NaN,NaN,2.386467,NaN,NaN,0.050000,NaN,NaN,...,NaN,0.037705,5.361814e-06,0.125000,0.034121,1.366994,9.854540,1.092000e+01,NaN,NaN



对应目标示例:


504    0.129412
503    0.031818
508   -0.066849
502   -0.045455
501   -0.102041
Name: RET, dtype: float64

## 结果小结与下一步
- 已删除高缺失特征 6 个，保留 88 个特征；删除缺失>90% 的行后仍有约 429 万行数据。
- 特征已中位数填补，当前 `X_imputed`、`y_step2` 无缺失，可直接进入标准化与时间序列CV。
- 为避免泄漏，后续标准化应在每个训练折内 `fit`，在验证/测试折仅 `transform`。
- 时间序列切分需用 `DATE`（以及可选 `permno` 作为ID）保留下来；收益可做双侧截尾减轻极端值影响。


In [4]:
# 保存分割索引并对收益截尾

def winsorize_series(s, lower=0.005, upper=0.995):
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi)

# DATE / permno 用于时间序列切分与标识
dates_for_split = work_df.loc[X_base.index, 'DATE'] if 'DATE' in work_df.columns else None
permno_for_id = work_df.loc[X_base.index, 'permno'] if 'permno' in work_df.columns else None

# 对收益做双侧截尾，减轻极端值影响
y_wins = winsorize_series(y_step2, lower=0.005, upper=0.995)

print("分割索引可用:", dates_for_split is not None)
if permno_for_id is not None:
    print("permno 已保留，可用于ID对齐")
print("截尾后目标示例:")
display(y_wins.head())


分割索引可用: True
permno 已保留，可用于ID对齐
截尾后目标示例:


504    0.129412
503    0.031818
508   -0.066849
502   -0.045455
501   -0.102041
Name: RET, dtype: float64

## 时间序列递归CV与基线模型（示例）
- 使用 `DATE` 做时间有序切分，避免前视偏差；示例采用 3 折扩展窗口（可调）。
- 先对样本下采样以便快速试跑（可关闭采样跑全量，但耗时大）。
- 每折：训练内 `StandardScaler` 拟合，验证集仅 transform；训练 OLS、Ridge、LASSO，输出 MSE、R²。
- 视需要调整折数、窗口比例、正则化超参网格。

In [ ]:
# 时间序列递归CV + 基线模型试跑（按月度滚动：最少训练120个月，验证窗口12个月）
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# 可调参数（按月）
use_date_window = True  # 可限制开发区间
dev_start = pd.Timestamp("2000-01-01")
dev_end   = pd.Timestamp("2020-12-31")


# 1) 日期窗口过滤（保持时间顺序）
if use_date_window:
    mask = (dates_for_split >= dev_start) & (dates_for_split <= dev_end)
    X_cv = X_base.loc[mask]
    y_cv = y_wins.loc[mask]
    dates_cv = dates_for_split.loc[mask]
else:
    X_cv = X_base
    y_cv = y_wins
    dates_cv = dates_for_split

# 确认 DATE 未进入特征
assert 'DATE' not in X_cv.columns, "DATE 不应作为特征参与回归"

# 2) 按日期排序
order = np.argsort(dates_cv.values)
X_cv = X_cv.iloc[order]
y_cv = y_cv.iloc[order]
dates_cv = dates_cv.iloc[order]

# 3) 构造月度时间折（Rolling固定10年训练 + 固定12个月验证）
months_cv = dates_cv.dt.to_period('M')
# 以 months_cv 为准生成 rolling folds（不要混用 months_full）
unique_months_cv = np.array(sorted(months_cv.unique()))

# 时间序列递归CV + 基线模型试跑（Rolling固定10年训练 + 固定12个月验证，开发窗口2000-2020）
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# 可调参数（按月）
use_date_window = True
dev_start = pd.Timestamp("2000-01-01")
dev_end   = pd.Timestamp("2020-12-31")

train_len = 120   # 10年 = 120个月
val_len   = 12    # 1年 = 12个月
step      = 12    # 每次向前滚动12个月（更省算力；想更细可改 1/3/6）

# 1) 日期窗口过滤（保持时间顺序）
if use_date_window:
    mask = (dates_for_split >= dev_start) & (dates_for_split <= dev_end)
    X_cv = X_base.loc[mask]
    y_cv = y_wins.loc[mask]
    dates_cv = dates_for_split.loc[mask]
else:
    X_cv = X_base
    y_cv = y_wins
    dates_cv = dates_for_split

# 确认 DATE 未进入特征
assert "DATE" not in X_cv.columns, "DATE 不应作为特征参与回归"

# 2) 按日期排序（保证时间顺序）
order = np.argsort(dates_cv.values)
X_cv = X_cv.iloc[order]
y_cv = y_cv.iloc[order]
dates_cv = dates_cv.iloc[order]

# 3) 构造月度时间折（Rolling固定10年训练 + 固定12个月验证）
months_cv = dates_cv.dt.to_period("M")
unique_months_cv = np.array(sorted(months_cv.unique()))

folds = []
start = train_len
while start + val_len <= len(unique_months_cv):
    train_month_list = unique_months_cv[start - train_len : start]
    val_month_list = unique_months_cv[start : start + val_len]

    train_mask = months_cv.isin(train_month_list)
    val_mask = months_cv.isin(val_month_list)

    if val_mask.sum() == 0:
        break

    folds.append((train_mask, val_mask))

    # 打印前两折区间，便于你肉眼确认切分正确
    if len(folds) <= 2:
        print(
            f"Fold {len(folds)}: "
            f"train {train_month_list[0]}~{train_month_list[-1]}, "
            f"val {val_month_list[0]}~{val_month_list[-1]}"
        )

    start += step

print(f"总折数: {len(folds)} (train={train_len}m, val={val_len}m, step={step}m)")

# 4) 逐折训练与评估（每折单独 fit imputer + scaler，避免数据泄露）
results = []
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "LASSO": Lasso(alpha=0.1, max_iter=3000),
}

for fold_id, (train_mask, val_mask) in enumerate(folds, 1):
    # 断言：训练集与验证集不重叠
    assert (train_mask & val_mask).sum() == 0, "train/val masks overlap!"

    # 断言：验证集覆盖的月份数正确（严格12个月；若你未来允许最后一折不足12个月，就改成 <= val_len）
    assert months_cv.loc[val_mask].nunique() == val_len, "val months != 12"

    X_train, X_val = X_cv.loc[train_mask], X_cv.loc[val_mask]
    y_train, y_val = y_cv.loc[train_mask], y_cv.loc[val_mask]

    # 折内缺失填补和标准化
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)
    X_val_imp = imputer.transform(X_val)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_imp)
    X_val_s = scaler.transform(X_val_imp)

    for name, model in models.items():
        model.fit(X_train_s, y_train)
        pred = model.predict(X_val_s)

        mse = mean_squared_error(y_val, pred)
        r2 = r2_score(y_val, pred)

        results.append(
            {
                "fold": fold_id,
                "model": name,
                "mse": mse,
                "r2": r2,
                "n_train": len(y_train),
                "n_val": len(y_val),
            }
        )
        print(
            f"Fold {fold_id:02d} {name}: "
            f"MSE={mse:.6f}, R2={r2:.4f}, n_train={len(y_train)}, n_val={len(y_val)}"
        )

results_df = pd.DataFrame(results)
display(results_df)


## 下一步：时间一致的正则化强度搜索（示例）
- 在同一外层时间折上，对 Ridge / LASSO 的 alpha 网格做评估，不使用内部 KFold，避免泄露。
- 每个 alpha 都在每折内独立 fit imputer+scaler，再评估验证集 MSE / R²。
- 选出平均 MSE 最低的 alpha 作为候选，供后续全量再训练或测试集评估。

In [ ]:
# 时间一致的 alpha 网格搜索（Ridge/LASSO）
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

ridge_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
lasso_grid = [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]

# 强制要求前一单元已经生成 folds / X_cv / y_cv
required = ["X_cv", "y_cv", "folds"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f"请先运行上一单元生成 {missing}，再运行本单元（避免folds不一致）")

def grid_search_cached_preprocess(alpha_list, model_cls, model_name):
    rows = []

    # 外层按 fold：每折只做一次 imputer/scaler
    for fold_id, (train_mask, val_mask) in enumerate(folds, 1):
        X_train, X_val = X_cv.loc[train_mask], X_cv.loc[val_mask]
        y_train, y_val = y_cv.loc[train_mask], y_cv.loc[val_mask]

        imputer = SimpleImputer(strategy="median")
        X_train_imp = imputer.fit_transform(X_train)
        X_val_imp = imputer.transform(X_val)

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train_imp)
        X_val_s = scaler.transform(X_val_imp)

        for alpha in alpha_list:
            if model_cls is Lasso:
                model = model_cls(alpha=alpha, max_iter=2000, tol=1e-3)  # 比5000更快，可再调
            else:
                model = model_cls(alpha=alpha)

            model.fit(X_train_s, y_train)
            pred = model.predict(X_val_s)

            rows.append({
                "model": model_name,
                "alpha": alpha,
                "fold": fold_id,
                "mse": mean_squared_error(y_val, pred),
                "r2": r2_score(y_val, pred),
                "n_train": len(y_train),
                "n_val": len(y_val),
            })

        print(f"{model_name}: fold {fold_id}/{len(folds)} finished")

    detail = pd.DataFrame(rows)
    summary = (detail.groupby(["model", "alpha"])
                     .agg(mse_mean=("mse","mean"),
                          mse_std=("mse","std"),
                          r2_mean=("r2","mean"),
                          r2_std=("r2","std"))
                     .reset_index()
                     .sort_values("mse_mean"))
    return detail, summary

ridge_detail, ridge_summary = grid_search_cached_preprocess(ridge_grid, Ridge, "Ridge")
print("Ridge alpha 网格结果：")
display(ridge_summary.reset_index(drop=True))

# LASSO 非常慢，建议先只跑3个alpha试跑确认流程没问题
lasso_grid_fast = [0.001, 0.01, 0.1]
lasso_detail, lasso_summary = grid_search_cached_preprocess(lasso_grid_fast, Lasso, "LASSO")
print("LASSO alpha 网格结果（快速版3个alpha）：")
display(lasso_summary.reset_index(drop=True))

## 全量时间序列评估（固定 alpha，更多样本）
- 使用全量日期（不裁剪窗口），保持时间有序递归折，折内独立 imputer+scaler，避免泄露。
- 采用网格搜索中略优的 LASSO alpha=0.001，Ridge alpha=1.0，作为固定基准。
- 增加折数（5 折）提升稳健性；可选留出末段作为最终测试。

In [ ]:
# 全量时间序列递归CV（按月度滚动：最少训练120个月，验证窗口12个月，含进度条）
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

print("==== 全量递归CV（按月度：train_min_months=120，val_months=12） ====")

# 使用全量数据，不做日期窗口裁剪
X_full = X_base
y_full = y_wins
full_dates = dates_for_split

# 按日期排序
order = np.argsort(full_dates.values)
X_full = X_full.iloc[order]
y_full = y_full.iloc[order]
full_dates = full_dates.iloc[order]

# 构造时间折（按月度：扩展训练 + 固定12个月验证）
months_full = full_dates.dt.to_period('M')
unique_months = np.sort(months_full.unique())
min_train_months = 120  # 10 年
val_months = 12         # 1 年
folds_full = []
start_idx = min_train_months
while start_idx < len(unique_months):
    val_start = start_idx
    val_end = min(start_idx + val_months, len(unique_months))
    train_cut = unique_months[val_start - 1]
    val_cut = unique_months[val_end - 1]
    train_months = unique_months[val_start - 120 : val_start]
    val_months = unique_months[val_start : val_start + 12]
    train_mask = months_cv.isin(train_months)
    val_mask = months_cv.isin(val_months)
    if val_mask.sum() == 0:
        break
    folds_full.append((train_mask, val_mask))
    start_idx = val_end  # 每次向前滚动一个验证窗口
    if fold_id <= 2:
        print(f"Fold {fold_id}: train {train_months[0]}~{train_months[-1]}, val {val_months[0]}~{val_months[-1]}")

print(f"全量折数: {len(folds_full)}，训练窗口≥{min_train_months}个月，验证窗口 {val_months} 个月")

ridge_alpha_final = 10.0
lasso_alpha_final = 0.005

results_full = []
for fold_id, (train_mask, val_mask) in enumerate(tqdm(folds_full, desc="Folds"), 1):
    X_train, X_val = X_full.loc[train_mask], X_full.loc[val_mask]
    y_train, y_val = y_full.loc[train_mask], y_full.loc[val_mask]

    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_val_imp = imputer.transform(X_val)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_imp)
    X_val_s = scaler.transform(X_val_imp)

    models = {
        "OLS": LinearRegression(),
        "Ridge": Ridge(alpha=ridge_alpha_final),
        "LASSO": Lasso(alpha=lasso_alpha_final, max_iter=5000)
    }

    for name, model in models.items():
        model.fit(X_train_s, y_train)
        pred = model.predict(X_val_s)
        mse = mean_squared_error(y_val, pred)
        r2 = r2_score(y_val, pred)
        results_full.append({"fold": fold_id, "model": name, "mse": mse, "r2": r2, "n_train": len(y_train), "n_val": len(y_val)})
        print(f"[Full] Fold {fold_id} {name}: MSE={mse:.6f}, R2={r2:.4f}, n_train={len(y_train)}, n_val={len(y_val)}")

results_full_df = pd.DataFrame(results_full)
display(results_full_df)


In [ ]:
# 汇总当前全量递归CV的 MSE / R2（按模型均值、标准差）
import pandas as pd

if 'results_full_df' not in globals() or results_full_df.empty:
    raise RuntimeError('请先运行全量递归CV单元以生成 results_full_df')

summary_full = (
    results_full_df
    .groupby('model')[['mse', 'r2']]
    .agg(['mean', 'std'])
    .reset_index()
)
print('全量递归CV汇总（5 折，train_min_frac=0.80）：')
display(summary_full)


In [ ]:
# 简单基线：常数预测（Zero 与 TrainMean），复用全量时间折
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

print("==== 基线评估（Zero / TrainMean） ====")

if 'folds_full' not in globals() or len(folds_full) == 0:
    raise RuntimeError("请先运行全量时间序列折构造单元（含 folds_full）")
if 'y_full' not in globals():
    raise RuntimeError("请先确保 y_full 已在全量评估单元中构造")

baseline_rows = []
for fold_id, (train_mask, val_mask) in enumerate(folds_full, 1):
    y_train = y_full.loc[train_mask].to_numpy()
    y_val = y_full.loc[val_mask].to_numpy()

    pred_zero = np.zeros(y_val.shape, dtype=float)
    mse_zero = mean_squared_error(y_val, pred_zero)
    r2_zero = r2_score(y_val, pred_zero)
    baseline_rows.append({"fold": fold_id, "model": "Zero", "mse": mse_zero, "r2": r2_zero, "n_train": len(y_train), "n_val": len(y_val)})

    mean_train = float(np.mean(y_train))
    pred_mean = np.full(y_val.shape, mean_train, dtype=float)
    mse_mean = mean_squared_error(y_val, pred_mean)
    r2_mean = r2_score(y_val, pred_mean)
    baseline_rows.append({"fold": fold_id, "model": "TrainMean", "mse": mse_mean, "r2": r2_mean, "n_train": len(y_train), "n_val": len(y_val)})

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

print("按模型聚合：")
display(baseline_df.groupby("model")[["mse", "r2"]].mean().reset_index())

print("线性模型 vs 基线（按模型聚合）：")
if 'results_full_df' in globals():
    combined_df = pd.concat([results_full_df, baseline_df], ignore_index=True)
    display(combined_df.groupby("model")[["mse", "r2"]].mean().reset_index())
else:
    print("results_full_df 不存在，请先运行全量递归CV单元")


In [ ]:
# 直接用当前配置训练全量模型并生成预测（LASSO alpha=0.005）
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from tqdm.auto import tqdm

print("==== 全量拟合 + 生成预测（谨慎：当前模型未优于基线） ====")

X_full = X_base
full_dates = dates_for_split
permno_full = permno_for_id if 'permno_for_id' in globals() else None

# 对齐排序
order = np.argsort(full_dates.values)
X_full = X_full.iloc[order]
full_dates = full_dates.iloc[order]
y_full_sorted = y_wins.iloc[order]
if permno_full is not None:
    permno_full = permno_full.iloc[order]

# 拟合全量 imputer + scaler + LASSO
imputer_final = SimpleImputer(strategy='median')
X_imp = imputer_final.fit_transform(X_full)
scaler_final = StandardScaler()
X_s = scaler_final.fit_transform(X_imp)

lasso_final = Lasso(alpha=0.005, max_iter=5000)
print("拟合模型...")
lasso_final.fit(X_s, y_full_sorted)

print("生成预测...")
pred_full = lasso_final.predict(X_s)

print("组装结果并保存...")
pred_df = pd.DataFrame({
    "DATE": full_dates.values,
    "pred_ret": pred_full
})
if permno_full is not None:
    pred_df.insert(1, "permno", permno_full.values)

# 进度条保存（按块写入，可选，这里用一条tqdm示意）
block_size = max(len(pred_df) // 10, 1)
with tqdm(total=len(pred_df), desc="Saving") as pbar:
    pred_df.to_csv("prediction_full_lasso.csv", index=False)
    pbar.update(len(pred_df))

print("预测完成，前几行：")
display(pred_df.head())
print("已保存 prediction_full_lasso.csv")
